In [ ]:
# Exportación para GEPHI 
# Un grafo por cada snapshot temporal

import pandas as pd
import networkx as nx
from pathlib import Path

# Define directories
gold_dir = Path("data/03_gold")
viz_dir = Path("viz/snapshots")

# Ensure the output directory exists
viz_dir.mkdir(parents=True, exist_ok=True)

def generate_clean_gexf(snapshot_key):
    edge_file = gold_dir / f"gold_edges_{snapshot_key}.parquet"
    node_file = gold_dir / f"gold_nodes_{snapshot_key}.parquet"
    
    # Quick sanity check to ensure the files for this date exist before reading
    if not edge_file.exists() or not node_file.exists():
        print(f"⚠️ Skipping snapshot {snapshot_key}: Parquet files not found.")
        return

    # Load strictly from Gold
    edges_df = pd.read_parquet(edge_file)
    nodes_df = pd.read_parquet(node_file)
    
    # Build graph object
    G = nx.from_pandas_edgelist(edges_df, source='source_node', target='target_node', create_using=nx.DiGraph())
    G.add_nodes_from(nodes_df['node_name'].tolist())
    
    # Map attributes safely
    for _, row in nodes_df.iterrows():
        node = row['node_name']
        if G.has_node(node):
            G.nodes[node]['pagerank'] = float(row['pagerank'])
            G.nodes[node]['in_degree'] = int(row['in_degree'])
            G.nodes[node]['is_in_bcc'] = str(row['is_in_lcc'])
            
    # Export using the verified directory path
    out_file = viz_dir / f"crystal_presentation_{snapshot_key}.gexf"
    nx.write_gexf(G, out_file)
    print(f"✅ Generated final Gephi asset: {out_file.name}")


# --- Generate Timeline and Process Snapshots ---

# 1. Anchor to your verified database state
anchor_date = "2026-05-06"

# 2. Generate exactly 30 snapshots spaced 100 days apart ending on the anchor
snapshot_dates = pd.date_range(end=anchor_date, periods=30, freq="100D")

print(f"Processing 30 snapshots from {snapshot_dates[0].strftime('%Y-%m-%d')} to {snapshot_dates[-1].strftime('%Y-%m-%d')}...\n")

# 3. Loop over the generated timestamps, format them as strings, and build the GEXF assets
for ts in snapshot_dates:
    snapshot_key = ts.strftime('%Y-%m-%d')
    generate_clean_gexf(snapshot_key)

print("\n🏁 All available network slices have been written to disk!")


Processing 30 snapshots from 2018-05-28 to 2026-05-06...



KeyError: 'is_in_bcc'